In [14]:
# import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [15]:
# create a SparkSession
pyspark = SparkSession.builder\
    .appName("DataQualityChecks")\
    .getOrCreate()

In [16]:
#load dataset
activity_df = pyspark.read.csv("C:/Users/jovit/Desktop/Netflix_Project/01.-data/03.curated/bronze/user_activity.csv", header=True, inferSchema=True)

In [17]:
#examine dataset
activity_df.show(5)
activity_df.printSchema()
activity_df.count()

+-----------+-------+-------+-------------+
|activity_id|user_id|show_id|watch_minutes|
+-----------+-------+-------+-------------+
|          1|    313|  s7076|          134|
|          2|    455|  s6205|          126|
|          3|    317|  s2292|           10|
|          4|    205|  s2100|          112|
|          5|    453|  s7494|          147|
+-----------+-------+-------+-------------+
only showing top 5 rows
root
 |-- activity_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- show_id: string (nullable = true)
 |-- watch_minutes: integer (nullable = true)



5000

In [18]:
#checking Missing Ids
activity_df.filter(col("activity_id").isNull()).show()
activity_df.filter(col("user_id").isNull()).show()
activity_df.filter(col("show_id").isNull()).show()

+-----------+-------+-------+-------------+
|activity_id|user_id|show_id|watch_minutes|
+-----------+-------+-------+-------------+
+-----------+-------+-------+-------------+

+-----------+-------+-------+-------------+
|activity_id|user_id|show_id|watch_minutes|
+-----------+-------+-------+-------------+
+-----------+-------+-------+-------------+

+-----------+-------+-------+-------------+
|activity_id|user_id|show_id|watch_minutes|
+-----------+-------+-------+-------------+
+-----------+-------+-------+-------------+



In [19]:
#Check duplicate records
activity_df.groupBy("activity_id")\
    .count()\
    .filter(col("count") > 1).show()

+-----------+-----+
|activity_id|count|
+-----------+-----+
+-----------+-----+



In [20]:
#Check invalid watch minutes
activity_df.filter(col("watch_minutes")<0).show()
activity_df.filter((col("watch_minutes")<0) | (col("watch_minutes")>300)).show()

+-----------+-------+-------+-------------+
|activity_id|user_id|show_id|watch_minutes|
+-----------+-------+-------+-------------+
+-----------+-------+-------+-------------+

+-----------+-------+-------+-------------+
|activity_id|user_id|show_id|watch_minutes|
+-----------+-------+-------+-------------+
+-----------+-------+-------+-------------+



In [21]:
#Derived Column
from pyspark.sql.functions import round
if activity_df is None:
    activity_df = pyspark.read.csv(
        "C:/Users/jovit/Desktop/Netflix_Project/01.-data/03.curated/user_activity.csv",
        header=True,
        inferSchema=True
    )
activity_df = activity_df.withColumn("watch_hours", round(col("watch_minutes") / 60, 2))
activity_df.show(5)

+-----------+-------+-------+-------------+-----------+
|activity_id|user_id|show_id|watch_minutes|watch_hours|
+-----------+-------+-------+-------------+-----------+
|          1|    313|  s7076|          134|       2.23|
|          2|    455|  s6205|          126|        2.1|
|          3|    317|  s2292|           10|       0.17|
|          4|    205|  s2100|          112|       1.87|
|          5|    453|  s7494|          147|       2.45|
+-----------+-------+-------+-------------+-----------+
only showing top 5 rows


In [22]:
# create clean datset
clean_activity_df =(
activity_df.dropDuplicates(["activity_id"])
.filter(col("activity_id").isNotNull())
.filter(col("user_id").isNotNull())
.filter(col("show_id").isNotNull())
.filter((col("watch_minutes") >= 0) & (col("watch_minutes") <= 300)))

In [24]:
# read the netflix_titles_clean.csv file into a DataFrame
users_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    "C:/Users/jovit/Desktop/Netflix_Project/01.-data/03.curated/bronze/users.csv"
) # read the users.csv file into a DataFrame
content_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    "C:/Users/jovit/Desktop/Netflix_Project/01.-data/03.curated/bronze/netflix_titles_clean.csv"
) 

In [25]:
# Clean users_df
clean_users_df = (
    users_df
    .dropDuplicates(["user_id"])
    .filter(col("user_id").isNotNull())
    .filter(col("user_name").isNotNull())
    .filter(col("country").isNotNull())
    .filter(col("subscription_type").isin("Basic", "Standard", "Premium"))
)

In [28]:
# Clean content_df
clean_content_df = (
    content_df
    .dropDuplicates(["show_id"])
    .filter(col("show_id").isNotNull())
    .filter(col("title").isNotNull())
    .filter(col("show_type").isin("Movie", "TV Show"))
    .filter(col("release_year").isNotNull())
)
clean_content_df.show(5)

+-------+---------+--------------------+--------------------+--------------------+-------------+----------+------------+------+--------+--------------------+--------------------+----------+
|show_id|show_type|               title|            director|           cast_name|      country|date_added|release_year|rating|duration|           listed_in|         description|added_year|
+-------+---------+--------------------+--------------------+--------------------+-------------+----------+------------+------+--------+--------------------+--------------------+----------+
|    s10|    Movie|        The Starling|      Theodore Melfi|Melissa McCarthy,...|United States|2021-09-24|        2021| PG-13| 104 min|    Comedies, Dramas|A woman adjusting...|    2021.0|
|  s1003|    Movie|        Tell Me When|      Gerardo Gatica|Jesús Zavala, Xim...|       Mexico|2021-04-21|        2021| TV-MA|  97 min|Comedies, Interna...|Workaholic Will p...|    2021.0|
|  s1009|    Movie|Motu Patlu VS Rob...|         S

In [41]:
import sys
print(sys.executable)


c:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\.venv\Scripts\python.exe


In [43]:
pip install pandas


  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl (12.5 MB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- --------------------------

In [49]:
pip show pandas

Name: pandas
Version: 3.0.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

 Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
 All rights reserved.

 Copyright (c) 2011-2026, Open source contributors.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are met:

 * Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

 * Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

 * Neither the name of the copyright holder nor the names of its
   contribut

In [51]:
import pandas as pd

pd.__version__

'3.0.3'

In [ ]:
import pandas as pd


In [58]:
#Save Silver Layer as CSV files
clean_users_df.toPandas().to_csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\users.csv",
    index=False
)

clean_activity_df.toPandas().to_csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\user_activity.csv",
    index=False
)

clean_content_df.toPandas().to_csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\netflix_titles.csv",
    index=False
)

In [62]:
# read the cleaned CSV files into DataFrames
clean_users_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\users.csv"
)

clean_activity_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\user_activity.csv"
)

clean_content_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\netflix_titles.csv"
)